# 02 DistilRoberta 训练

对齐 `code/distil_train_exp927.py`，用于训练二分类 DistilRoberta。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import roc_auc_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

BASE_MODEL = ROOT / 'models/distil/base_distilroberta'
if not BASE_MODEL.exists():
    BASE_MODEL = ROOT / 'resources/distilroberta_bundle/DATASETS/distilroberta-base'
if not BASE_MODEL.exists():
    BASE_MODEL = Path('distilroberta-base')

TRAIN_CSV = ROOT / 'code/train1.csv'
VALID_CSV = ROOT / 'code/nonTargetText_llm_slightly_modified_gen.csv'
OUTPUT_DIR = ROOT / 'models/distil/distilroberta-finetuned_v927_notebook'

MAX_LENGTH = 128
BATCH_SIZE = 2
GRAD_ACC = 8
NUM_EPOCHS = 16.0
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
SEED = 42

print('BASE_MODEL =', BASE_MODEL)

In [ ]:
def softmax_np(logits):
    x = logits - np.max(logits, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)


def load_binary_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if not {'text', 'label'}.issubset(df.columns):
        raise ValueError(f'{path} must contain text,label columns')
    out = df[['text', 'label']].copy()
    out['text'] = out['text'].fillna('').astype(str).str.strip('\n')
    out['label'] = out['label'].astype(int)
    return out


train = load_binary_csv(TRAIN_CSV)
valid = load_binary_csv(VALID_CSV)
print('train:', train.shape, 'valid:', valid.shape)

## 数据分布

在编码前查看标签与长度分布。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
train_counts = train['label'].value_counts().sort_index()
valid_counts = valid['label'].value_counts().sort_index()
ax[0].bar(train_counts.index.astype(str), train_counts.values, color=['#4c72b0', '#dd8452'])
ax[0].set_title('Distil Train Label Distribution')
ax[1].bar(valid_counts.index.astype(str), valid_counts.values, color=['#4c72b0', '#dd8452'])
ax[1].set_title('Distil Valid Label Distribution')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'distil_label_distribution.png', dpi=220)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(train['text'].str.len(), bins=60, stat='density', alpha=0.35, label='train', ax=ax)
sns.histplot(valid['text'].str.len(), bins=60, stat='density', alpha=0.35, label='valid', ax=ax)
ax.set_title('Distil Text Length Distribution')
ax.set_xlabel('Character Length')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'distil_text_length_distribution.png', dpi=220)
plt.show()

In [ ]:
ds_train = Dataset.from_pandas(train)
ds_valid = Dataset.from_pandas(valid)

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL))

def preprocess(examples):
    return tokenizer(examples['text'], max_length=MAX_LENGTH, padding=True, truncation=True)

ds_train_enc = ds_train.map(preprocess, batched=True)
ds_valid_enc = ds_valid.map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(str(BASE_MODEL), num_labels=2)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    auc = roc_auc_score(labels, probs[:, 1], multi_class='ovr')
    return {'roc_auc': auc}

In [ ]:
train_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_torch',
    fp16=torch.cuda.is_available(),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model='roc_auc',
    report_to='none',
    save_total_limit=2,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=ds_train_enc,
    eval_dataset=ds_valid_enc,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('saved:', OUTPUT_DIR)

## 训练后评估

训练完成后立即绘制 loss/AUC/ROC/混淆矩阵。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix

sns.set_theme(style='whitegrid', context='talk')
PLOT_DIR = OUTPUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

log_df = pd.DataFrame(trainer.state.log_history)
display(log_df.tail(10))

# 1) 训练/验证 loss
fig, ax = plt.subplots(figsize=(10, 6))
if 'loss' in log_df.columns:
    x_train = log_df.loc[log_df['loss'].notna(), 'step']
    y_train = log_df.loc[log_df['loss'].notna(), 'loss']
    ax.plot(x_train, y_train, label='train loss', lw=2)
if 'eval_loss' in log_df.columns:
    x_eval = log_df.loc[log_df['eval_loss'].notna(), 'step']
    y_eval = log_df.loc[log_df['eval_loss'].notna(), 'eval_loss']
    ax.plot(x_eval, y_eval, label='eval loss', lw=2)
ax.set_title('DistilRoberta Training Curve (Loss)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'distil_loss_curve.png', dpi=220)
plt.show()

# 2) 验证 AUC 随训练变化
if 'eval_roc_auc' in log_df.columns and log_df['eval_roc_auc'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 6))
    x = log_df.loc[log_df['eval_roc_auc'].notna(), 'epoch']
    y = log_df.loc[log_df['eval_roc_auc'].notna(), 'eval_roc_auc']
    ax.plot(x, y, marker='o', lw=2, color='#dd8452')
    ax.set_title('DistilRoberta Validation AUC by Epoch')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ROC-AUC')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'distil_auc_curve.png', dpi=220)
    plt.show()

# 3) 验证集 ROC + 混淆矩阵
pred_out = trainer.predict(ds_valid_enc)
logits = pred_out.predictions
labels = pred_out.label_ids
probs = softmax_np(logits)[:, 1]

fpr, tpr, _ = roc_curve(labels, probs)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, lw=2, label=f'AUC={roc_auc:.4f}')
ax.plot([0, 1], [0, 1], '--', color='gray')
ax.set_title('DistilRoberta ROC on Validation Set')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'distil_valid_roc.png', dpi=220)
plt.show()

cm = confusion_matrix(labels, (probs >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('DistilRoberta Confusion Matrix @0.5')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'distil_valid_confusion_matrix.png', dpi=220)
plt.show()

print('plots saved to', PLOT_DIR)